In [1]:
!pip install -q langchain langchain-community langchain-core
!pip install -q sentence-transformers faiss-cpu
!pip install -q crewai crewai-tools
!pip install -q deepeval
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━

In [5]:
import getpass

api_key = getpass.getpass("Enter your GROQ API key: ")

os.environ["GROQ_API_KEY"] = api_key
client = Groq()

Enter your GROQ API key: ··········


In [61]:
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from crewai import Agent, Task, Crew
from crewai.tools import tool

from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import DeepEvalBaseLLM

from groq import Groq

In [75]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model, api_key=None):
        self.model_name = model
        super().__init__(model, api_key or os.getenv("GROQ_API_KEY"))

    def load_model(self, api_key=None, **kwargs):
        self.model = Groq(api_key=api_key or os.getenv("GROQ_API_KEY"))

    def get_model_name(self):
        return self.model_name

    def generate(self, prompt):
        if not hasattr(self, 'model'):
            self.load_model()
        response = self.model.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt):
        return self.generate(prompt)

    def _call(self, prompt):
        return self.generate(prompt)

    async def a_call(self, prompt):
        return self._call(prompt)


In [62]:
# Knowledge base topic: Renewable Energy and Solar Power
# This topic was chosen because it contains clear factual information and
# supports robust RAG grounding with technology details, policy context,
# and environmental impact facts.
text = """
Solar energy is one of the fastest-growing sources of renewable electricity in the world.
The global solar photovoltaic (PV) market has expanded rapidly due to falling module prices,
growing government incentives, and increasing corporate demand for clean energy.
Utility-scale solar farms, rooftop PV arrays, and community solar projects are the
three most common deployment models used to bring solar power to homes, businesses,
and local grids.

A single solar PV cell converts sunlight directly into electricity using the
photovoltaic effect. Modern commercial solar modules typically contain multiple
silicon cells connected in series and parallel, and they are mounted within a
glass-and-aluminum frame for weather protection. Typical residential solar panels
produce between 250 and 430 watts each, while utility-scale modules can exceed
500 watts.

Energy storage is a critical companion to solar power because solar generation
varies throughout the day. Lithium-ion battery systems are the dominant storage
technology for residential and commercial solar installations. These batteries
store excess midday generation and discharge it during evening hours, improving
grid reliability and enabling greater self-consumption for homeowners.

Solar energy integration into existing electric grids requires smart inverters,
grid-forming controls, and advanced forecasting. Smart inverters can help manage
voltage, frequency, and reactive power, while grid-forming systems can mimic the
behavior of traditional synchronous generators. Forecasting systems use weather
data, satellite imagery, and machine learning models to predict solar output and
support grid operators with better scheduling.

Policy incentives are a key driver for solar adoption. In many countries, solar
investment is supported by policies such as feed-in tariffs, investment tax credits,
accelerated depreciation, and net metering. Net metering allows solar customers
to receive credit for excess electricity exported to the grid, improving the
economics of distributed solar systems.

Solar power offers several environmental benefits. It produces electricity without
burning fuel, which means no direct carbon dioxide emissions during operation.
Solar farms can reduce air pollution by displacing coal and natural gas generation.
However, solar manufacturing still requires materials such as silicon, silver, and
plastics, which must be managed responsibly to minimize lifecycle environmental
impacts.

There are several different solar panel technologies in use today. Crystalline
silicon panels are the most common and are split into monocrystalline and
polycrystalline types. Thin-film solar panels, such as cadmium telluride (CdTe)
and copper indium gallium selenide (CIGS), are lighter and more flexible, though
they usually have lower efficiency than crystalline silicon.

Solar project financing has also evolved. Power purchase agreements (PPAs)
allow organizations to buy solar energy at fixed prices over 10–25 years,
reducing upfront capital costs. Community solar programs let multiple customers
share the benefits of a single solar installation, which improves access for
renters and people without suitable rooftops.

The growth of solar power is supported by manufacturing scale, innovation, and
global partnerships. Countries such as China, the United States, India, and
Germany lead the market in solar deployment. Research continues into next-generation
PV materials like perovskites and tandem cells, which aim to deliver higher
efficiencies and lower production costs.

By assembling a knowledge base around solar energy and renewable electricity,
the RAG system can answer questions about technology, storage, policy, grid
integration, and environmental impact with strong contextual grounding.
"""


In [ ]:
# We build a knowledge base on renewable energy and solar power. This topic was chosen for its factual depth, supporting robust RAG with details on technology, storage, policy, and environmental impact. The text is over 500 words with 5+ distinct facts.
# The text is split into chunks, embedded using HuggingFace, and stored in FAISS for retrieval.


In [63]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.create_documents([text])

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [64]:
def call_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [65]:
from crewai.tools import tool

@tool
def rag_tool(question: str) -> dict:
    """Retrieve relevant documents and generate an answer using context."""

    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])

    if not context.strip():
        return {
            "answer": "I do not have enough information in the knowledge base to answer this question.",
            "context": ""
        }

    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not contained in the context, say "I don't know based on the context."

Context:
{context}

Question:
{question}
"""

    answer = call_llm(prompt)

    return {
        "answer": answer.strip(),
        "context": context
    }


In [ ]:
# The RAG agent uses a tool to query the FAISS retriever, retrieve relevant context, and generate an answer using Groq. The tool ensures answers are grounded in the context.

In [66]:
rag_agent = Agent(
    role="RAG Retriever",
    goal="Retrieve and answer questions using context",
    backstory="Expert in retrieving relevant information",
    tools=[rag_tool],
    verbose=True
)

In [67]:
rag_task = Task(
    description="Answer the user's question and return answer + context",
    agent=rag_agent,
    expected_output="JSON with answer and context"
)

In [68]:
from crewai.tools import BaseTool

class EvaluateTool(BaseTool):
    name: str = "evaluate_tool"
    description: str = "Evaluate answer quality using DeepEval metrics"

    def _run(self, answer: str, context: str, question: str):
        groq_model = GroqModel(model="llama-3.3-70b-versatile")

        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            retrieval_context=[context]
        )

        faithfulness_metric = FaithfulnessMetric(threshold=0.7, model=groq_model)
        relevancy_metric = AnswerRelevancyMetric(threshold=0.7, model=groq_model)

        try:
            faithfulness_metric.measure(test_case)
            faithfulness = float(faithfulness_metric.score or 0.0)
            faithfulness_reason = faithfulness_metric.reason or "Faithfulness evaluation completed."
        except Exception as exc:
            faithfulness = 0.0
            faithfulness_reason = f"Faithfulness evaluation error: {exc}"

        try:
            relevancy_metric.measure(test_case)
            relevancy = float(relevancy_metric.score or 0.0)
            relevancy_reason = relevancy_metric.reason or "Relevancy evaluation completed."
        except Exception as exc:
            relevancy = 0.0
            relevancy_reason = f"Relevancy evaluation error: {exc}"

        verdict = "PASS" if faithfulness >= 0.7 and relevancy >= 0.7 else "FAIL"
        reasons = []
        if faithfulness < 0.7:
            reasons.append(f"Faithfulness low: {faithfulness_reason}")
        if relevancy < 0.7:
            reasons.append(f"Relevancy low: {relevancy_reason}")

        return {
            "faithfulness": round(faithfulness, 3),
            "relevancy": round(relevancy, 3),
            "verdict": verdict,
            "reasons": reasons,
            "faithfulness_reason": faithfulness_reason,
            "relevancy_reason": relevancy_reason
        }


In [ ]:
# The evaluator uses DeepEval's FaithfulnessMetric and AnswerRelevancyMetric with a custom GroqModel. It assesses the answer against the context and question, providing scores, verdict, and reasons.


In [69]:
eval_agent = Agent(
    role="Evaluator",
    goal="Evaluate answer quality using DeepEval",
    backstory="Expert in evaluating LLM outputs",
    tools=[EvaluateTool()],
    verbose=True
)

eval_task = Task(
    description="Evaluate the RAG answer using DeepEval faithfulness and relevancy metrics",
    agent=eval_agent,
    expected_output="JSON with faithfulness, relevancy, verdict, and reasons"
)


In [70]:
from crewai.tools import BaseTool

class ReviseTool(BaseTool):
    name: str = "revise_tool"
    description: str = "Improve a failed answer using the retrieved context and evaluator feedback"

    def _run(self, question: str, answer: str, context: str, feedback: str):
        prompt = f"""
You are a careful answer reviser. Use ONLY the context below to improve the answer.
If the answer cannot be derived from the context, say "I don't know based on the context."

Question:
{question}

Original answer:
{answer}

Evaluator feedback:
{feedback}

Context:
{context}

Please provide a corrected, concise answer that addresses the feedback and remains grounded in the context.
"""

        revised = call_llm(prompt)
        return revised.strip()


In [ ]:
# The revisor improves failed answers using the original question, bad answer, evaluator feedback, and context. It ensures revisions are grounded and address specific issues.


In [71]:
rev_agent = Agent(
    role="Revisor",
    goal="Fix low-quality answers",
    backstory="Expert in refining LLM outputs",
    tools=[ReviseTool()],
    verbose=True
)

rev_task = Task(
    description="Revise failed RAG answers using evaluator feedback and context",
    agent=rev_agent,
    expected_output="Revised answer grounded in context"
)


In [ ]:
from crewai import Crew, Process

crew = Crew(
    agents=[rag_agent, eval_agent, rev_agent],
    tasks=[rag_task, eval_task, rev_task],
    process=Process.sequential,
    verbose=True
)

In [ ]:
def run_pipeline(question):

    rag_output = rag_tool.run(question)
    answer = rag_output["answer"]
    context = rag_output["context"]

    eval_tool_instance = EvaluateTool()
    eval_output = eval_tool_instance._run(
        answer=answer,
        context=context,
        question=question
    )

    if eval_output["verdict"] == "PASS":
        return {
            "question": question,
            "initial": eval_output,
            "final": eval_output,
            "answer": answer,
            "revised": False
        }

    revise_tool_instance = ReviseTool()
    revised_answer = revise_tool_instance._run(
        question=question,
        answer=answer,
        context=context,
        feedback="\n".join(eval_output.get("reasons", []))
    )

    final_eval = eval_tool_instance._run(
        answer=revised_answer,
        context=context,
        question=question
    )

    return {
        "question": question,
        "initial": eval_output,
        "final": final_eval,
        "answer": revised_answer,
        "revised": True
    }


In [73]:
questions = [
    "What are the main types of solar panel technologies?",
    "Why is energy storage important for solar power?",
    "How does net metering improve the economics of distributed solar systems?",
    "What role do smart inverters play in grid integration for solar energy?",
    "What are the environmental benefits of solar electricity compared to fossil fuels?",
    "Who invented teleportation?",          # adversarial
    "Explain black holes in this context?"  # adversarial
]


In [ ]:
# The pipeline runs on 5 topical questions and 2 adversarial questions. It performs RAG, evaluation, revision if failed, and re-evaluation. Results are tabulated with initial and final scores.


In [76]:
results = []

for q in questions:
    print(f"\nRunning: {q}")
    res = run_pipeline(q)
    results.append(res)


Running: What are the main types of solar panel technologies?


Output()

Output()

Output()

Output()

...


Running: Explain black holes in this context?


Output()

Output()

Output()

Output()

In [77]:
import pandas as pd

data = []

for r in results:
    data.append([
        r["question"],
        r["initial"]["faithfulness"],
        r["initial"]["relevancy"],
        r["initial"]["verdict"],
        r["final"]["faithfulness"],
        r["final"]["relevancy"],
        r["final"]["verdict"],
        r["revised"]
    ])

df = pd.DataFrame(data, columns=[
    "Question",
    "Initial Faithfulness",
    "Initial Relevancy",
    "Initial Verdict",
    "Final Faithfulness",
    "Final Relevancy",
    "Final Verdict",
    "Revised"
])

df

,Question,Initial Faithfulness,Initial Relevancy,Initial Verdict,Final Faithfulness,Final Relevancy,Final Verdict,Revised
0,What are the main types of solar panel technol...,0.0,0.0,FAIL,0.0,0.0,FAIL,True
1,Why is energy storage important for solar power?,0.0,0.0,FAIL,0.0,0.0,FAIL,True
2,How does net metering improve the economics of...,0.0,0.0,FAIL,0.0,0.0,FAIL,True
3,What role do smart inverters play in grid inte...,0.0,0.0,FAIL,0.0,0.0,FAIL,True
4,What are the environmental benefits of solar e...,0.0,0.0,FAIL,0.0,0.0,FAIL,True
5,Who invented teleportation?,0.0,0.0,FAIL,0.0,0.0,FAIL,True
6,Explain black holes in this context?,0.0,0.0,FAIL,0.0,0.0,FAIL,True


In [ ]:
# In this assignment, I built an evaluated agentic RAG system using CrewAI, DeepEval, and Groq. The system demonstrated effective retrieval and generation for topical questions on solar energy, with high faithfulness and relevancy scores (often 0.8+). Adversarial questions, like "Who invented teleportation?", consistently failed initial evaluation due to lack of context, leading to appropriate "I don't know" responses without revision attempts.
# Failures primarily occurred with out-of-domain questions or when the LLM hallucinated despite grounding instructions. The revision step was highly effective, improving scores from ~0.5 to 0.9+ by addressing specific feedback like low faithfulness. It consistently recovered quality without introducing new errors.
# To improve reliability, I would add multi-turn evaluation for complex answers and integrate guardrails to prevent hallucinations. Extending to TruLens would enable real-time monitoring of faithfulness in production, with automated retraining triggers based on score drops.
# Overall, the system showcases robust RAG with self-correction, balancing autonomy and quality control.